# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout, 5-fold stratified CV on training rows, selection by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
print(sha('data.csv')); print(sha('holdout.csv'))
df=pd.read_csv('data.csv'); hold=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==7000 and len(hold)==1400 and len(set(hold))==1400

## Data and features
`device_serial` is excluded (identifier). The other features (channel, device age, price, reported daily use at registration) are all known at sale/registration, i.e. before the outcome.

In [ ]:
TARGET='claim_within_year'
features=['channel','device_age_years','price_eur','daily_hours']
mask=np.zeros(len(df),bool); mask[hold]=True
train=df[~mask]; test=df[mask]
Xtr,ytr=train[features],train[TARGET]; Xte,yte=test[features],test[TARGET]
print(len(train),len(test),ytr.mean(),yte.mean())

## Candidates (named before any scoring)
Preprocessing is inside each pipeline, so it is fit within CV folds.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def pre(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
candidates={
 'baseline_prior':Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'random_forest':Pipeline([('p',pre(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0))]),
 'hist_gb':Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=0)
cv_scores={n:cross_val_score(p,Xtr,ytr,cv=cv,scoring='roc_auc').tolist() for n,p in candidates.items()}
summary=pd.Series({n:np.mean(s) for n,s in cv_scores.items()}).sort_values(ascending=False)
print(summary)
chosen=summary.index[0]; print('chosen:',chosen)

## Refit chosen candidate on all training rows; score holdout once

In [ ]:
final=candidates[chosen].fit(Xtr,ytr)
holdout_auc=roc_auc_score(yte,final.predict_proba(Xte)[:,1])
print('holdout roc_auc',holdout_auc)

In [ ]:
cv_folds=[[[int(i) for i in train.index[a]],[int(i) for i in train.index[b]]] for a,b in cv.split(Xtr,ytr)]
evidence={'features':features,'candidates':list(candidates),'cv_folds':cv_folds,'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':[int(i) for i in hold],'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))